# 06 | Training Loop & Text Generation
## Sprint 6 — Day 3

**Previous notebook:** `05` — a complete tiny GPT with a working forward pass.
This is the final notebook. We train the model and watch it generate text.

---

We will train on a small text corpus (Shakespeare) using a standard
next-token prediction objective: given tokens `[t_0, t_1, ..., t_{n-1}]`,
predict `[t_1, t_2, ..., t_n]`. The loss is cross-entropy over the
vocabulary at every position.

The training loop is pure PyTorch — no Trainer, no callbacks, no abstractions.
After training we will sample from the model autoregressively and visualise
what it learned to attend to.

---

**By the end of this notebook you will have:**
- Implemented cross-entropy loss for language modelling
- Written a complete PyTorch training loop from scratch
- Trained the tiny GPT on a real text corpus
- Generated text from the trained model
- Plotted the training loss curve and visualised learned attention patterns

---

*This is the final notebook. You have built a Transformer from scratch.*

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import urllib.request
import os

torch.manual_seed(42)

d_model     = 64
num_heads   = 8
d_ff        = 256
num_blocks  = 4
max_seq_len = 128
batch_size  = 16
seq_len     = 64
lr          = 3e-4
num_steps   = 3000

def positional_encoding(max_seq_len, d_model):
    PE = torch.zeros(max_seq_len, d_model)
    position = torch.arange(0, max_seq_len).unsqueeze(1).float()
    div_term = torch.pow(10000.0, torch.arange(0, d_model, 2).float() / d_model)
    PE[:, 0::2] = torch.sin(position / div_term)
    PE[:, 1::2] = torch.cos(position / div_term)
    return PE

class InputEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_seq_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pe = positional_encoding(max_seq_len, d_model)
    def forward(self, x):
        return self.embedding(x) + self.pe[:x.shape[1], :]

class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta  = nn.Parameter(torch.zeros(d_model))
        self.eps   = eps
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std  = x.std(dim=-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
    def forward(self, x):
        return self.linear2(F.relu(self.linear1(x)))

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, V), weights

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
    def make_causal_mask(self, T):
        return torch.tril(torch.ones(T, T))
    def forward(self, x):
        B, T, _ = x.shape
        Q = self.W_Q(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        mask = self.make_causal_mask(T)
        attn_out, weights = scaled_dot_product_attention(Q, K, V, mask)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.W_O(attn_out), weights

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.ff        = FeedForward(d_model, d_ff)
        self.norm1     = LayerNorm(d_model)
        self.norm2     = LayerNorm(d_model)
    def forward(self, x):
        attn_out, weights = self.attention(x)
        x = self.norm1(x + attn_out)
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)
        return x, weights

class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_blocks, max_seq_len):
        super().__init__()
        self.embedding = InputEmbedding(vocab_size, d_model, max_seq_len)
        self.blocks    = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff)
            for _ in range(num_blocks)
        ])
        self.norm    = LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
    def forward(self, x):
        x = self.embedding(x)
        all_weights = []
        for block in self.blocks:
            x, weights = block(x)
            all_weights.append(weights)
        x = self.norm(x)
        return self.lm_head(x), all_weights

print("All components loaded.")

**LOAD AND TOKENIZE *"SHAKESPEARE"***

In [ ]:
url  = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
path = "../shakespeare.txt"

if not os.path.exists(path):
    urllib.request.urlretrieve(url, path)
    print("Downloaded shakespeare.txt")
else:
    print("shakespeare.txt already exists")

with open(path, "r") as f:
    text = f.read()

chars    = sorted(set(text))
vocab_size = len(chars)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [char_to_idx[c] for c in s]
decode = lambda l: "".join([idx_to_char[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

split = int(0.9 * len(data))
train_data = data[:split]
val_data   = data[split:]

print(f"Corpus length:  {len(text):,} characters")
print(f"Vocabulary size: {vocab_size} unique characters")
print(f"Training tokens: {len(train_data):,}")
print(f"Validation tokens: {len(val_data):,}")
print(f"Sample vocab: {chars[:20]}")
print(f"Expected initial loss: {torch.log(torch.tensor(float(vocab_size))):.4f}")

**BATCH SAMPLER**

In [ ]:
def get_batch(data, batch_size, seq_len):
    ix = torch.randint(0, len(data) - seq_len - 1, (batch_size,))
    x  = torch.stack([data[i:i + seq_len] for i in ix])
    y  = torch.stack([data[i + 1:i + seq_len + 1] for i in ix])
    return x, y

x_sample, y_sample = get_batch(train_data, batch_size, seq_len)
print(f"Input batch shape:  {x_sample.shape}")
print(f"Target batch shape: {y_sample.shape}")
print(f"\nSample input  (decoded): '{decode(x_sample[0].tolist()[:30])}'")
print(f"Sample target (decoded): '{decode(y_sample[0].tolist()[:30])}'")

**TRAINING LOOP**

In [ ]:
model     = GPT(vocab_size, d_model, num_heads, d_ff, num_blocks, max_seq_len)
optimiser = torch.optim.Adam(model.parameters(), lr=lr)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Starting training for {num_steps} steps...\n")

train_losses = []
log_every    = 100

for step in range(num_steps):
    x, y = get_batch(train_data, batch_size, seq_len)

    logits, _ = model(x)

    loss = F.cross_entropy(
        logits.view(-1, vocab_size),
        y.view(-1)
    )

    optimiser.zero_grad()
    loss.backward()
    optimiser.step()

    train_losses.append(loss.item())

    if step % log_every == 0:
        print(f"Step {step:4d} | Loss: {loss.item():.4f}")

print(f"\nFinal loss: {train_losses[-1]:.4f}")

**PLOTTING LOST CURVE**

In [ ]:
smoothed = np.convolve(train_losses, np.ones(50)/50, mode='valid')

plt.figure(figsize=(12, 5))
plt.plot(train_losses, alpha=0.3, color='steelblue', label='Raw loss')
plt.plot(range(49, len(train_losses)), smoothed, color='steelblue', linewidth=2, label='Smoothed (50-step avg)')
plt.axhline(y=np.log(vocab_size), color='red', linestyle='--', alpha=0.7, label=f'Random baseline (log {vocab_size} = {np.log(vocab_size):.2f})')
plt.xlabel('Training Step')
plt.ylabel('Cross-Entropy Loss')
plt.title('Training Loss Curve — Tiny GPT on Shakespeare')
plt.legend()
plt.tight_layout()
plt.show()

**TEXT GENERATION**

In [ ]:
def generate(model, seed_text, max_new_tokens, temperature=0.8):
    model.eval()
    tokens = encode(seed_text)
    tokens = torch.tensor(tokens, dtype=torch.long).unsqueeze(0)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            tokens_cropped = tokens[:, -max_seq_len:]
            logits, _      = model(tokens_cropped)
            next_logits    = logits[0, -1, :]
            next_logits    = next_logits / temperature
            probs          = F.softmax(next_logits, dim=-1)
            next_token     = torch.multinomial(probs, num_samples=1)
            tokens         = torch.cat([tokens, next_token.unsqueeze(0)], dim=1)

    model.train()
    return decode(tokens[0].tolist())

output = generate(model, seed_text="KING:", max_new_tokens=300, temperature=0.8)
print(output)

**VISUALIZING TRAINED ATTENTION PATTERNS**

In [ ]:
model.eval()
sample_text    = "To be or not to be that is the question"
sample_tokens  = torch.tensor(encode(sample_text), dtype=torch.long).unsqueeze(0)

with torch.no_grad():
    _, all_weights = model(sample_tokens)

fig, axes = plt.subplots(num_blocks, num_heads, figsize=(24, 14))

for block_idx in range(num_blocks):
    for head_idx in range(num_heads):
        ax = axes[block_idx, head_idx]
        w  = all_weights[block_idx][0, head_idx].numpy()
        im = ax.imshow(w, cmap='viridis', aspect='auto', vmin=0, vmax=1)
        ax.set_title(f'B{block_idx+1} H{head_idx+1}', fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle('Trained Attention Patterns — All Blocks, All Heads', fontsize=13)
plt.tight_layout()
plt.show()

**COMPARING GENERATION AT DIFFERENT TEMPERATURES**

In [ ]:
for temp in [0.5, 0.8, 1.2]:
    print(f"\n{'='*50}")
    print(f"Temperature: {temp}")
    print('='*50)
    output = generate(model, seed_text="KING:", max_new_tokens=150, temperature=temp)
    print(output)

# Sprint 6 Summary — Training Loop & Text Generation

---

## What This Sprint Built

A complete training pipeline for the tiny GPT — cross-entropy loss, a pure PyTorch
training loop, character-level tokenisation, and autoregressive text generation with
temperature sampling. The model was trained on real Shakespeare text and generates
Shakespeare-flavoured output entirely from scratch.

---

## Concept 1 — Cross-Entropy Loss for Language Modelling

The training objective is **next-token prediction**: given tokens $[t_0, t_1, ..., t_{T-1}]$,
predict $[t_1, t_2, ..., t_T]$. Every position tries to predict the next one.

For one position with logits $z \in \mathbb{R}^V$ and correct token index $y$:

$$\mathcal{L} = -\log\left(\frac{e^{z_y}}{\displaystyle\sum_{j=1}^{V} e^{z_j}}\right) = -\log\, P(\text{correct token})$$

Averaged over all $B \times T$ predictions in a batch:

$$\mathcal{L}_{total} = -\frac{1}{B \cdot T} \sum_{b=1}^{B} \sum_{t=1}^{T} \log\, P\!\left(t_{t+1}^{(b)} \mid t_0^{(b)}, ..., t_t^{(b)}\right)$$

**The random baseline:** at initialisation, a model assigns equal probability to all $V$ tokens.
Expected starting loss $= \log(V)$. For $V=65$: $\log(65) \approx 4.17$.

| Loss value | What it means |
|---|---|
| $\approx \log(V) = 4.17$ | Random — model knows nothing |
| $\approx 2.5$ | Learned character frequencies and common bigrams |
| $\approx 2.0$ | Learned word-level patterns and dialogue structure |
| $\approx 1.3$ | GPT-2 small on character-level Shakespeare |

---

## Concept 2 — The Training Loop

Five steps, repeated every iteration:

```
1. FORWARD PASS    logits, _ = model(x)
2. COMPUTE LOSS    loss = F.cross_entropy(logits.view(-1, V), y.view(-1))
3. ZERO GRADIENTS  optimiser.zero_grad()
4. BACKWARD PASS   loss.backward()
5. UPDATE WEIGHTS  optimiser.step()
```

**Why `zero_grad()` is necessary:** PyTorch accumulates gradients by default.
Without zeroing, each step adds to gradients from all previous steps — weights
update based on a blend of all past gradients, not the current batch. Always zero
before each backward pass.

**Why Adam:** Adam tracks a running average of both gradient and squared gradient
per parameter, giving each parameter its own adaptive learning rate. Converges
much faster than plain SGD on Transformer models. Learning rate `3e-4` is the
standard starting point for Transformer training with Adam.

**logits reshape:** `F.cross_entropy` expects shape `(N, C)` for predictions and
`(N,)` for targets. We reshape $(B, T, V) \to (B \cdot T, V)$ and $(B, T) \to (B \cdot T,)$
to satisfy this — every token prediction treated independently.

---

## Concept 3 — Autoregressive Generation

Text is generated one token at a time. The model's own output becomes its next input:

```
seed:    "KING:"
step 1:  model("KING:") → probs → sample → "B"  → "KING:B"
step 2:  model("KING:B") → probs → sample → "y" → "KING:By"
step 3:  model("KING:By") → probs → sample → " " → "KING:By "
...
```

**Temperature sampling** controls the distribution sharpness before sampling:

$$P_\tau(\text{token}) = \text{softmax}(z\, /\, \tau)$$

| $\tau$ | Effect | Behaviour |
|---|---|---|
| $< 1.0$ (e.g. 0.5) | Sharper distribution | More repetitive, more "certain" |
| $= 1.0$ | Model's natural distribution | Standard sampling |
| $> 1.0$ (e.g. 1.2) | Flatter distribution | More creative, more chaotic |

**Context window:** `tokens[:, -max_seq_len:]` crops the growing sequence to the
last `max_seq_len` tokens. The model has no memory beyond this window — identical
to how GPT-3 and GPT-4 work with their context windows.

**`torch.multinomial`** samples from the probability distribution — not argmax.
This introduces variety: high-probability tokens are likely but not guaranteed,
preventing the degenerate repetition that greedy decoding produces.

---

## Reading Your Training Results

### Loss progression

```
Step    0 | Loss: 4.3139   ← near random baseline (4.17) ✓
Step  100 | Loss: 2.8154   ← fast early learning: character frequencies
Step  500 | Loss: 2.4032   ← bigrams, common words
Step 1000 | Loss: 2.2710   ← word patterns, punctuation habits
Step 2000 | Loss: 2.1138   ← dialogue structure, speaker labels
Step 3000 | Loss: 1.9924   ← Shakespeare-flavoured patterns
```

Final loss `1.99` is **2.18 units below the random baseline** — the model learned
substantial structure from the corpus with only 207,360 parameters.

### Why the early loss drops so fast

Low-hanging fruit is grabbed first: space frequency, common letters (`e`,`t`,`a`),
capital letters after newlines. These patterns reduce loss significantly with
minimal weight change — strong, consistent gradient signal from every batch.

### The repetition collapse (`lllll`, `t t t t`)

A known failure mode of small language models. After predicting `l`, the model
has seen many `ll` pairs in training (`shall`, `will`) — so `l→l` has high
learned probability. At low temperature the distribution sharpens further,
locking the model into repetition. Larger models escape this through longer-range
dependency modelling.

---

## Reading the Trained Attention Patterns

Comparing trained patterns (Sprint 6) to random patterns (Sprint 3):

| Observation | Random (Sprint 3) | Trained (Sprint 6) |
|---|---|---|
| Upper triangle | Arbitrary noise | Perfectly dark — causal mask held |
| Bright cells | Scattered randomly | Concentrated at meaningful positions |
| Head diversity | Different noise patterns | Different *learned* specialisations |
| Diagonal | No consistent pattern | Many heads show strong diagonal (recent token attention) |

**Across blocks:** early blocks (B1, B2) show more distributed, local attention.
Later blocks (B3, B4) show sharper, more selective patterns — consistent with
research showing early blocks learn surface features and later blocks learn
higher-level structure.

**Yellow-green cells** (weight ≈ 1.0) in later blocks indicate heads that learned
to attend almost exclusively to specific position types — this specificity is
entirely absent in the untrained model.

---

## The Complete Architecture — Final View

```
Input: token indices          (B, T)
           ↓
InputEmbedding                (B, T, 64)    Sprint 1
  ├── nn.Embedding            learned token meaning
  └── Sinusoidal PE           fixed position fingerprint
           ↓
TransformerBlock × 4          (B, T, 64)    Sprints 2–4
  ├── MultiHeadAttention                    Sprint 3
  │     ├── 8 parallel heads
  │     ├── Causal mask                     Sprint 5
  │     └── W_O projection
  ├── Add & Norm (residual + LayerNorm)      Sprint 4
  ├── FeedForward (64→256→64)               Sprint 4
  └── Add & Norm
           ↓
Final LayerNorm               (B, T, 64)    Sprint 5
           ↓
LM Head: Linear(64, 65)       (B, T, 65)    Sprint 5
           ↓
Output: logits                (B, T, 65)
```

---

## What Scale Buys

Your model: 207K parameters, 3000 training steps → loss ≈ 1.99, garbled output

| Model | Params | Architecture | Loss |
|---|---|---|---|
| This notebook | 207K | 4 blocks, d=64, h=8 | ~1.99 |
| GPT-2 small | 117M | 12 blocks, d=768, h=12 | ~1.3 |
| GPT-2 large | 774M | 36 blocks, d=1280, h=20 | ~1.2 |

The **architecture is identical**. The difference is entirely scale and training duration.
Every mechanism you implemented — attention, residuals, layer norm, causal masking —
is present unchanged in GPT-2, GPT-3, and GPT-4.

---

## Key Takeaways

- Cross-entropy loss measures $-\log P(\text{correct token})$ — lower is better, zero is impossible
- The random baseline $\log(V)$ is a concrete sanity check for initialisation
- The five-step training loop is universal: forward → loss → zero\_grad → backward → step
- `logits.view(-1, V)` and `y.view(-1)` are the standard reshape for sequence cross-entropy
- Autoregressive generation feeds the model's own output back as input, one token at a time
- Temperature $\tau < 1$ sharpens the distribution; $\tau > 1$ flattens it
- Repetition collapse is a small-model failure mode — large models resolve it with depth
- Trained attention patterns are structured and diverse; random patterns are not
- The architecture you built is identical to GPT-2 — scale is the only difference

---

## Project Complete

You have built a Transformer from scratch:

| Sprint | What you built |
|---|---|
| 01 | Sinusoidal positional encoding + token embeddings |
| 02 | Scaled dot-product attention |
| 03 | Multi-head attention with parallel heads |
| 04 | Full Transformer block: Add & Norm + FFN + residuals |
| 05 | Tiny GPT: causal masking + stacked blocks + LM head |
| 06 | Training loop + text generation + visualisation |

Every component was implemented from scratch in PyTorch.
Every formula was derived before the code was written.
Every output was explained, not just observed.